# World of Shadow Work — full offline factory (Colab Pro+, A100)

Run on a **GPU A100** runtime (Runtime → Change runtime type → A100), then **Runtime → Run all**.

Builds the asset bundle: corpus → CLIP embeddings → UMAP galaxy + k-NN webs → image-to-3D
**vessel** keyframes (LGM) → packed `vessel.wswv` — then pushes the assets back to GitHub so a
`git pull` on your Mac picks them up (relaunch, no code change).

**Reuse your existing corpus (recommended).** AIC's image server throttles re-fetches (HTTP 403),
so don't re-download — put the corpus you already have on Google Drive and point `DRIVE_CORPUS`
at it (folder or `corpus.zip`). Set `USE_DRIVE_CORPUS = True`. Edit **CONFIG**, then Run All.
Vessels: `VESSEL_LIMIT = 24` first to smoke-test, then 0 (= all) for the full run.

In [ ]:
# ===== CONFIG =====
GH_TOKEN   = ""        # GitHub token (repo scope) to push assets back. Blank => download manually.
REPO       = "9LiveZZZ-Git/MAT201B_Projects"
BRANCH     = "world-of-shadow-work"

# Corpus: reuse what you already downloaded (avoids AIC 403). Upload your local
# reagency/factory/corpus to Drive as a folder OR a corpus.zip, and point here:
USE_DRIVE_CORPUS = True
DRIVE_CORPUS     = "/content/drive/MyDrive/wosw/corpus.zip"   # a .zip OR a folder on your Drive
CORPUS_PER_QUERY = 20   # only used if USE_DRIVE_CORPUS = False (re-fetch; may hit 403)

# Optional: bulk up the galaxy with Open Images V7 (CC BY 2.0; PHOTOS, not museum art).
# Great for labor/agency (tools, hands, machines); magic stays museum-sourced.
USE_OPENIMAGES = False
OI_PER_CLASS   = 200

VESSEL_MODE  = "all"    # "all" = one vessel per image; "reps" = one per cluster (fast)
VESSEL_LIMIT = 0        # cap vessel inputs (0 = no cap; try 24 first to smoke-test)
VESSEL_G     = 8000     # gaussians per vessel keyframe (stage_d caps the total bank size)

In [ ]:
!nvidia-smi -L

In [ ]:
import os
_auth = (GH_TOKEN + "@") if GH_TOKEN else ""
!git clone -b {BRANCH} https://{_auth}github.com/{REPO}.git
%cd MAT201B_Projects/reagency/factory

In [ ]:
!pip -q install open_clip_torch umap-learn faiss-cpu hdbscan scikit-learn plyfile pillow

In [ ]:
# Corpus: reuse the local one from Drive (no re-fetch -> no AIC 403), or fetch fresh.
import os, glob, shutil
if USE_DRIVE_CORPUS:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -rf corpus _cz
    if DRIVE_CORPUS.endswith('.zip'):
        !unzip -q -o "{DRIVE_CORPUS}" -d _cz
        shutil.move('_cz/corpus' if os.path.isdir('_cz/corpus') else '_cz', 'corpus')
    else:
        os.symlink(DRIVE_CORPUS, 'corpus')
    print('corpus images:', len(glob.glob('corpus/images/**/*.jpg', recursive=True)))
else:
    !python3 fetch_corpus.py --per-query {CORPUS_PER_QUERY}

In [ ]:
# Optional: augment with a themed, capped Open Images V7 subset (CC BY 2.0). Photos, not museum
# art — diversifies the galaxy but shifts the archival look. (magic stays museum-only.)
if USE_OPENIMAGES:
    !pip -q install fiftyone
    !python3 fetch_openimages.py --per-class {OI_PER_CLASS}

In [ ]:
!python3 stage_a_embed.py        # CLIP ViT-L/14 image+text embedding (A100)

In [ ]:
!python3 stage_b_layout.py       # UMAP galaxy + kNN webs + clusters + cluster_reps + atlas -> ../assets

In [ ]:
!python3 prep_vessel_inputs.py --mode {VESSEL_MODE} --limit {VESSEL_LIMIT}

In [ ]:
# LGM image-to-3D (3DTopia/LGM). torch + CUDA are preinstalled on Colab.
!pip -q install -U xformers
![ -d diff-gaussian-rasterization ] || git clone --recursive https://github.com/ashawkey/diff-gaussian-rasterization
!pip -q install ./diff-gaussian-rasterization
!pip -q install git+https://github.com/NVlabs/nvdiffrast
![ -d LGM ] || git clone https://github.com/3DTopia/LGM
!cd LGM && pip -q install -r requirements.txt
!mkdir -p LGM/pretrained
![ -f LGM/pretrained/model_fp16.safetensors ] || wget -q -O LGM/pretrained/model_fp16.safetensors https://huggingface.co/ashawkey/LGM/resolve/main/model_fp16_fixrot.safetensors

In [ ]:
import os
inp = os.path.abspath("work/vessel_inputs")
out = os.path.abspath("work/vessels")
os.makedirs(out, exist_ok=True)
%cd LGM
!python infer.py big --resume pretrained/model_fp16.safetensors --workspace {out} --test_path {inp}
%cd ..
!echo "generated:" && ls work/vessels | wc -l

In [ ]:
!python3 stage_d_vessel.py --plys work/vessels --G {VESSEL_G}
!ls -lh ../assets

In [ ]:
# Push the generated assets so your Mac can `git pull` them. (Needs GH_TOKEN in CONFIG.)
!git -C .. add assets
!git -C .. -c user.email="wosw@colab" -c user.name="wosw-colab" commit -m "assets: galaxy + vessels (Colab)"
!git -C .. push
# No token? download instead:  from google.colab import files; files.download('../assets/vessel.wswv')